# 1\.2\.3 TLC temporal aggregation

The TLC datasets are a different beast than the traffic and bus datasets\. While those datasets arrive in a partially aggregated form, TLC records every individual trip\. Across the retained Yellow Taxi, Green Taxi, and FHVHV datasets, that adds up to nearly one billion trips\. An earlier attempt to simply add temporal fields and carry the trip\-level records forward quickly exposed the downside of that approach: large storage requirements, long processing times, and little practical benefit for the analyses we actually plan to perform\.

Rather than creating multiple generations of massive intermediate files, this notebook standardizes and aggregates the TLC data in a single pass\. Each trip is assigned its canonical pickup\-based timestamp, enriched with the project's shared temporal dimensions, and aggregated into Taxi Zone\-level mobility summaries\. Because the downstream anomaly detection, clustering, and mobility regime analyses all operate on aggregated mobility states rather than individual trips, this approach dramatically reduces data volume while preserving the information we actually need\.

One design choice worth calling out is that we retain the hour\-level representation rather than aggregating directly into temporal buckets\. Temporal buckets can always be constructed later, but once hourly detail is lost it cannot be recovered\. Keeping the hour gives us maximum flexibility for future aggregation strategies while still supporting the project's standard temporal bucket framework\.


In [1]:
from pathlib import Path
import gc

import duckdb
import pandas as pd
import pyarrow.parquet as pq


In [2]:
SOURCE_DATA_DIR = Path("source_data")
PIPELINE_DATA_DIR = Path("pipeline_data")

TLC_INTERMEDIATE_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.2.3.intermediate_tables"
TLC_FINAL_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.2.3.final_tables"

SOURCE_DATA_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DATA_DIR.mkdir(parents=True, exist_ok=True)
TLC_INTERMEDIATE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TLC_FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Master Control Panel

RUN_RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC = False
RUN_TLC_MOBILITY_TABLE_GENERATION = False
FILTER_INVALID_TLC_TRIPS = True
RUN_TLC_MOBILITY_METRIC_QUALITY_VALIDATION = False

MAX_VALID_TLC_TRIP_DURATION_SECONDS = 86_400
MIN_VALID_TLC_TRIP_DURATION_SECONDS = 60
MIN_VALID_TLC_TRIP_DISTANCE = 0
MAX_VALID_TLC_TRIP_DISTANCE = 100
MAX_VALID_TLC_TRIP_SPEED_MPH = 100

## 1\.2\.3\.1  Generate TLC Spatiotemporal Mobility Tables

This section turns raw TLC trip records into compact spatiotemporal mobility tables\. We add the temporal and spatial fields we need, then aggregate immediately so we do not carry nearly one billion trip\-level records through the rest of the pipeline\.

In [4]:
# ---------------------------------------------------------------------
# Configure TLC spatiotemporal mobility table generation
# ---------------------------------------------------------------------

TLC_MOBILITY_OUTPUT_DIR = TLC_FINAL_OUTPUT_DIR

TLC_TEMPORAL_VALIDATION_MANIFEST_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_temporal_field_validation_manifest.csv"
)

DUCKDB_TEMP_DIR = TLC_INTERMEDIATE_OUTPUT_DIR / "duckdb_temp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

TLC_DATA_DIR = SOURCE_DATA_DIR / "tlc"

TLC_MOBILITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DUCKDB_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

retained_tlc_datasets = {
    "yellow": {
        "pickup_datetime_col": "tpep_pickup_datetime",
        "dropoff_datetime_col": "tpep_dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_distance",
        "duration_seconds_expr": """
            DATE_DIFF(
                'second',
                tpep_pickup_datetime,
                tpep_dropoff_datetime
            )
        """,
    },
    "green": {
        "pickup_datetime_col": "lpep_pickup_datetime",
        "dropoff_datetime_col": "lpep_dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_distance",
        "duration_seconds_expr": """
            DATE_DIFF(
                'second',
                lpep_pickup_datetime,
                lpep_dropoff_datetime
            )
        """,
    },
    "fhvhv": {
        "pickup_datetime_col": "pickup_datetime",
        "dropoff_datetime_col": "dropoff_datetime",
        "pickup_zone_col": "PULocationID",
        "dropoff_zone_col": "DOLocationID",
        "distance_col": "trip_miles",
        "duration_seconds_expr": "trip_time",
    },
}

retained_tlc_datasets

{'yellow': {'pickup_datetime_col': 'tpep_pickup_datetime',
  'dropoff_datetime_col': 'tpep_dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col': 'trip_distance',
  'duration_seconds_expr': "\n            DATE_DIFF(\n                'second',\n                tpep_pickup_datetime,\n                tpep_dropoff_datetime\n            )\n        "},
 'green': {'pickup_datetime_col': 'lpep_pickup_datetime',
  'dropoff_datetime_col': 'lpep_dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col': 'trip_distance',
  'duration_seconds_expr': "\n            DATE_DIFF(\n                'second',\n                lpep_pickup_datetime,\n                lpep_dropoff_datetime\n            )\n        "},
 'fhvhv': {'pickup_datetime_col': 'pickup_datetime',
  'dropoff_datetime_col': 'dropoff_datetime',
  'pickup_zone_col': 'PULocationID',
  'dropoff_zone_col': 'DOLocationID',
  'distance_col

Before rebuilding the TLC mobility tables, we first inspect the raw trip\-level distance, duration, and implied speed distributions\. This diagnostic helps quantify how many records would be excluded by the updated quality filters and confirms that extreme derived speeds are caused by implausible source distance/duration combinations rather than downstream aggregation logic\.

In [5]:
# ---------------------------------------------------------------------
# Diagnose raw TLC distance, duration, and implied speed quality
# ---------------------------------------------------------------------

RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "raw_tlc_distance_duration_diagnostic.csv"
)

if RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH.exists() and not RUN_RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC:

    raw_tlc_distance_duration_diagnostic_df = pd.read_csv(
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

    print(
        "Loaded existing raw TLC distance/duration diagnostic:",
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

else:

    raw_quality_results = []

    # This diagnostic scans monthly raw TLC files and writes progress after each file.
    # That matters because the raw TLC universe is huge and a full scan can be interrupted.
    if RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH.exists():

        existing_raw_quality_df = pd.read_csv(
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

        processed_keys = set(
            zip(
                existing_raw_quality_df["dataset_type"],
                existing_raw_quality_df["service_month"],
            )
        )

        raw_quality_results.append(existing_raw_quality_df)

        print(
            "Resuming existing raw TLC distance/duration diagnostic:",
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

    else:

        processed_keys = set()

        print(
            "Starting new raw TLC distance/duration diagnostic:",
            RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
        )

    for dataset_type, config in retained_tlc_datasets.items():

        distance_col = config["distance_col"]
        duration_seconds_expr = config["duration_seconds_expr"]

        source_files = sorted(
            TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
        )

        print(
            f"\nDiagnosing {dataset_type}: {len(source_files):,} monthly parquet files"
        )

        for file_index, source_path in enumerate(source_files, start=1):

            service_month = (
                source_path.name
                .split("_tripdata_")[1]
                .replace(".parquet", "")
            )

            diagnostic_key = (dataset_type, service_month)

            if diagnostic_key in processed_keys:

                print(
                    f"Skipping {dataset_type} {service_month} "
                    f"({file_index:,}/{len(source_files):,}) — already diagnosed"
                )

                continue

            print(
                f"Diagnosing {dataset_type} {service_month} "
                f"({file_index:,}/{len(source_files):,})"
            )

            # These checks evaluate the raw trip universe before aggregation.
            # The goal is to quantify impossible trips, not to remove anything in this diagnostic step.
            query = f"""
                SELECT
                    '{dataset_type}' AS dataset_type,
                    '{service_month}' AS service_month,

                    COUNT(*) AS raw_trip_rows,

                    SUM(CASE WHEN "{distance_col}" < 0 THEN 1 ELSE 0 END)
                        AS negative_distance_rows,

                    SUM(CASE WHEN "{distance_col}" > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
                        AS excessive_distance_rows,

                    SUM(CASE WHEN {duration_seconds_expr} < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                        AS short_duration_rows,

                    SUM(CASE WHEN {duration_seconds_expr} > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                        AS excessive_duration_rows,

                    SUM(
                        CASE
                            WHEN {duration_seconds_expr} > 0
                             AND ("{distance_col}" / ({duration_seconds_expr} / 3600.0)) > {MAX_VALID_TLC_TRIP_SPEED_MPH}
                            THEN 1 ELSE 0
                        END
                    ) AS excessive_speed_rows,

                    MIN("{distance_col}") AS min_distance,
                    QUANTILE_CONT("{distance_col}", 0.50) AS p50_distance,
                    QUANTILE_CONT("{distance_col}", 0.95) AS p95_distance,
                    QUANTILE_CONT("{distance_col}", 0.99) AS p99_distance,
                    QUANTILE_CONT("{distance_col}", 0.999) AS p999_distance,
                    MAX("{distance_col}") AS max_distance,

                    MIN({duration_seconds_expr}) AS min_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.50) AS p50_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.95) AS p95_duration_seconds,
                    QUANTILE_CONT({duration_seconds_expr}, 0.99) AS p99_duration_seconds,
                    MAX({duration_seconds_expr}) AS max_duration_seconds

                FROM read_parquet('{source_path.as_posix()}')
            """

            monthly_quality_df = duckdb.sql(query).df()

            raw_quality_results.append(monthly_quality_df)

            raw_tlc_distance_duration_diagnostic_df = (
                pd.concat(raw_quality_results, ignore_index=True)
                .drop_duplicates(
                    subset=["dataset_type", "service_month"],
                    keep="last"
                )
                .sort_values(["dataset_type", "service_month"])
                .reset_index(drop=True)
            )

            raw_tlc_distance_duration_diagnostic_df.to_csv(
                RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH,
                index=False
            )

            processed_keys.add(diagnostic_key)

            print(
                f"Saved diagnostic progress through {dataset_type} {service_month}"
            )

    raw_tlc_distance_duration_diagnostic_df = pd.read_csv(
        RAW_TLC_DISTANCE_DURATION_DIAGNOSTIC_PATH
    )

display(raw_tlc_distance_duration_diagnostic_df)

Loaded existing raw TLC distance/duration diagnostic: pipeline_data/1.2.3.intermediate_tables/raw_tlc_distance_duration_diagnostic.csv


,dataset_type,service_month,raw_trip_rows,negative_distance_rows,excessive_distance_rows,short_duration_rows,excessive_duration_rows,excessive_speed_rows,min_distance,p50_distance,p95_distance,p99_distance,p999_distance,max_distance,min_duration_seconds,p50_duration_seconds,p95_duration_seconds,p99_duration_seconds,max_duration_seconds
0,fhvhv,2023-01,18479031,0.0,1699.0,2238.0,0.0,6.0,0.0,2.897,16.060,25.79,51.150000,407.563,0,906.0,2526.0,3685.00,35359
1,fhvhv,2023-02,17960971,0.0,1266.0,1638.0,0.0,6.0,0.0,2.860,15.791,25.81,50.120000,423.000,0,914.0,2565.0,3740.00,43234
2,fhvhv,2023-03,20413539,0.0,1676.0,2162.0,0.0,8.0,0.0,2.913,16.320,26.69,52.200000,307.190,0,940.0,2744.0,4053.00,51226
3,fhvhv,2023-04,19144903,0.0,1571.0,1865.0,0.0,12.0,0.0,2.990,16.464,27.08,52.330000,362.190,0,946.0,2752.0,4094.00,63519
4,fhvhv,2023-05,19847676,0.0,1826.0,1790.0,1.0,6.0,0.0,3.066,16.860,27.49,53.444625,320.970,0,990.0,2985.0,4472.00,94853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,yellow,2025-12,4305006,0.0,197.0,105134.0,44.0,974.0,0.0,1.810,12.700,19.55,30.400000,322576.170,-2684,876.0,2877.0,4795.95,516712
114,yellow,2026-01,3724889,0.0,162.0,83747.0,35.0,732.0,0.0,1.810,12.450,19.54,29.790000,269097.480,-702,802.0,2565.0,4286.00,450474
115,yellow,2026-02,3399866,0.0,172.0,73605.0,37.0,620.0,0.0,1.800,12.050,19.41,29.920000,328522.200,-63,841.0,2627.0,4351.00,555909
116,yellow,2026-03,3952451,0.0,161.0,89190.0,28.0,930.0,0.0,1.820,12.700,19.58,29.900000,288381.680,-11200,797.0,2680.0,4515.00,457236


Findings\. The diagnostic confirms that the TLC source data contains a small but persistent tail of invalid trip records, especially implausible distances and implied speeds\. The core distributions look reasonable — for example, yellow taxi p99 distances are generally around 19–21 miles and FHVHV p99 distances are around 25–28 miles — but the maximum distances are clearly impossible, often reaching tens or hundreds of thousands of miles\. Short\-duration records are also common, especially in yellow and green taxi files\. This supports the new filtering rules: keep the valid majority of TLC trips, but remove trips with excessive distance, very short duration, excessive duration, or impossible implied speed before aggregation\.

Before applying the new TLC quality filters, we summarize the results of the raw distance, duration, and implied\-speed diagnostic\. Records were flagged if they contained trips shorter than 60 seconds, trips longer than 24 hours, trip distances greater than 100 miles, or implied speeds greater than 100 mph\. This diagnostic helps quantify the scale of data quality issues and validate that the proposed filtering rules target a relatively small number of clearly implausible records\.

In [6]:
# ---------------------------------------------------------------------
# Summarize raw TLC distance, duration, and speed diagnostics
# ---------------------------------------------------------------------

raw_tlc_quality_summary_df = (
    raw_tlc_distance_duration_diagnostic_df
    .groupby("dataset_type", as_index=False)
    .agg(
        raw_trip_rows=("raw_trip_rows", "sum"),
        negative_distance_rows=("negative_distance_rows", "sum"),
        excessive_distance_rows=("excessive_distance_rows", "sum"),
        short_duration_rows=("short_duration_rows", "sum"),
        excessive_duration_rows=("excessive_duration_rows", "sum"),
        excessive_speed_rows=("excessive_speed_rows", "sum"),
        max_distance=("max_distance", "max"),
        min_duration_seconds=("min_duration_seconds", "min"),
        max_duration_seconds=("max_duration_seconds", "max"),
    )
)

raw_tlc_quality_summary_df["total_flagged_rows"] = (
    raw_tlc_quality_summary_df[
        [
            "negative_distance_rows",
            "excessive_distance_rows",
            "short_duration_rows",
            "excessive_duration_rows",
            "excessive_speed_rows",
        ]
    ]
    .sum(axis=1)
)

raw_tlc_quality_summary_df["pct_flagged_rows"] = (
    raw_tlc_quality_summary_df["total_flagged_rows"]
    / raw_tlc_quality_summary_df["raw_trip_rows"]
).round(4)

display(raw_tlc_quality_summary_df)

,dataset_type,raw_trip_rows,negative_distance_rows,excessive_distance_rows,short_duration_rows,excessive_duration_rows,excessive_speed_rows,max_distance,min_duration_seconds,max_duration_seconds,total_flagged_rows,pct_flagged_rows
0,fhvhv,778424569,0.0,71518.0,64259.0,1.0,237.0,5380.78,0,94853,136015.0,0.0002
1,green,2160506,0.0,1047.0,67750.0,2.0,8060.0,278990.28,-3420,90046,76859.0,0.0356
2,yellow,143080418,0.0,6418.0,2337843.0,915.0,45748.0,398608.62,-1694897908,892846,2390924.0,0.0167


Findings\. The diagnostic shows that data quality issues are concentrated in the Taxi datasets, particularly Yellow Taxi\. FHVHV data is remarkably clean, with only about 0\.02% of records flagged despite nearly 778 million trips\. Yellow Taxi contains approximately 2\.4 million flagged records \(1\.67%\), driven primarily by extremely short trip durations and implausible implied speeds\. Green Taxi shows the highest flagged rate at 3\.56%, although the dataset is much smaller\. Across all three datasets, the problematic records represent a small minority of observations, suggesting that targeted filtering should substantially improve metric quality while preserving the vast majority of trip activity\.

Now we process one monthly file at a time\. For each file, DuckDB derives the shared temporal fields, standardizes pickup and dropoff Taxi Zone identifiers, and aggregates individual trips into origin\-destination mobility states\. Each resulting record represents the number of trips and average trip characteristics observed for a specific Pickup Taxi Zone × Dropoff Taxi Zone × Date × Hour combination\. Preserving both pickup and dropoff zones allows us to retain mobility flow patterns while dramatically reducing the volume of data carried into downstream analysis\. If the notebook gets interrupted, we can rerun this cell and it will skip outputs that already exist\.

In [7]:
# ---------------------------------------------------------------------
# Generate TLC spatiotemporal mobility tables
# ---------------------------------------------------------------------

tlc_mobility_jobs = []

for dataset_type, config in retained_tlc_datasets.items():

    source_files = sorted(
        TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
    )

    for source_path in source_files:

        service_month = (
            source_path.name
            .split("_tripdata_")[1]
            .replace(".parquet", "")
        )

        output_path = (
            TLC_MOBILITY_OUTPUT_DIR /
            f"{dataset_type}_tripdata_{service_month}_mobility.parquet"
        )

        output_exists = (
            output_path.exists()
            and output_path.stat().st_size > 0
        )

        tlc_mobility_jobs.append({
            "dataset_type": dataset_type,
            "service_month": service_month,
            "source_path": source_path,
            "output_path": output_path,
            "output_exists": output_exists,
            **config,
        })

missing_tlc_mobility_jobs = [
    job for job in tlc_mobility_jobs
    if not job["output_exists"]
]

print("Expected TLC mobility outputs:", len(tlc_mobility_jobs))
print("Missing TLC mobility outputs:", len(missing_tlc_mobility_jobs))

if (
    RUN_TLC_MOBILITY_TABLE_GENERATION
    or len(missing_tlc_mobility_jobs) > 0
):

    total_jobs = len(tlc_mobility_jobs)

    for i, job in enumerate(tlc_mobility_jobs, start=1):

        dataset_type = job["dataset_type"]
        service_month = job["service_month"]
        source_path = job["source_path"]
        output_path = job["output_path"]

        pickup_datetime_col = job["pickup_datetime_col"]
        pickup_zone_col = job["pickup_zone_col"]
        dropoff_zone_col = job["dropoff_zone_col"]
        distance_col = job["distance_col"]
        duration_seconds_expr = job["duration_seconds_expr"]

        if RUN_TLC_MOBILITY_TABLE_GENERATION and output_path.exists():
            output_path.unlink()

        if (
            output_path.exists()
            and output_path.stat().st_size > 0
            and not RUN_TLC_MOBILITY_TABLE_GENERATION
        ):
            print(
                f"[{i}/{total_jobs}] Already exists, skipping: "
                f"{output_path.name}",
                flush=True
            )
            continue

        print(
            f"[{i}/{total_jobs}] Processing {dataset_type} {service_month}: "
            f"{source_path.name}",
            flush=True
        )

        # Filtering happens before aggregation so bad trip-level records cannot contaminate
        # average distance, duration, or implied-speed metrics in the mobility states.
        trip_quality_filter = f"""
            {duration_seconds_expr} BETWEEN {MIN_VALID_TLC_TRIP_DURATION_SECONDS}
                AND {MAX_VALID_TLC_TRIP_DURATION_SECONDS}
            AND "{distance_col}" BETWEEN {MIN_VALID_TLC_TRIP_DISTANCE}
                AND {MAX_VALID_TLC_TRIP_DISTANCE}
            AND (
                "{distance_col}" / ({duration_seconds_expr} / 3600.0)
            ) <= {MAX_VALID_TLC_TRIP_SPEED_MPH}
        """

        where_clause = (
            f"WHERE {trip_quality_filter}"
            if FILTER_INVALID_TLC_TRIPS
            else ""
        )

        # This query standardizes each trip to pickup-based time and origin-destination grain,
        # then immediately aggregates to avoid carrying trip-level TLC records downstream.
        query = f"""
            COPY (
                SELECT
                    '{dataset_type}' AS dataset_type,

                    CAST("{pickup_zone_col}" AS INTEGER) AS pickup_zone_id,
                    CAST("{dropoff_zone_col}" AS INTEGER) AS dropoff_zone_id,

                    CAST("{pickup_datetime_col}" AS DATE) AS date,
                    EXTRACT(year FROM "{pickup_datetime_col}") AS year,
                    EXTRACT(month FROM "{pickup_datetime_col}") AS month,
                    CAST(
                        STRFTIME("{pickup_datetime_col}", '%w')
                        AS INTEGER
                    ) AS day_of_week,
                    EXTRACT(hour FROM "{pickup_datetime_col}") AS hour,

                    CASE
                        WHEN STRFTIME("{pickup_datetime_col}", '%w') IN ('0', '6')
                            THEN 'weekend_'
                        ELSE 'weekday_'
                    END ||
                    CASE
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 6 AND 9
                            THEN 'am_peak'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 10 AND 15
                            THEN 'midday'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 16 AND 19
                            THEN 'pm_peak'
                        WHEN EXTRACT(hour FROM "{pickup_datetime_col}") BETWEEN 20 AND 23
                            THEN 'evening'
                        ELSE 'overnight'
                    END AS temporal_bucket,

                    CASE
                        WHEN CAST("{pickup_datetime_col}" AS DATE) >= DATE '2025-01-05'
                            THEN 'post_cp'
                        ELSE 'pre_cp'
                    END AS pre_post_cp,

                    COUNT(*) AS trip_count,
                    AVG("{distance_col}") AS avg_trip_distance,
                    AVG({duration_seconds_expr}) AS avg_trip_duration_seconds

                FROM read_parquet(
                    '{source_path.as_posix()}'
                )

                {where_clause}

                GROUP BY
                    dataset_type,
                    pickup_zone_id,
                    dropoff_zone_id,
                    date,
                    year,
                    month,
                    day_of_week,
                    hour,
                    temporal_bucket,
                    pre_post_cp
            )
            TO '{output_path.as_posix()}'
            (FORMAT PARQUET)
        """

        con = duckdb.connect()

        try:
            con.execute(f"SET temp_directory='{DUCKDB_TEMP_DIR.as_posix()}'")
            con.execute("SET memory_limit='3GB'")
            con.execute(query)

        finally:
            con.close()

        del query
        gc.collect()

        print(
            f"    Wrote: {output_path.name}",
            flush=True
        )

else:

    print(
        "All TLC mobility outputs already exist. "
        "Nothing to process."
    )

Expected TLC mobility outputs: 118
Missing TLC mobility outputs: 0
All TLC mobility outputs already exist. Nothing to process.


Before doing formal validation, let’s just look at what we created\. This gives us a quick gut\-check on the output grain, the fields we carried forward, and whether the aggregated mobility table looks like the object we intended to build\.

In [8]:
# ---------------------------------------------------------------------
# Inspect sample TLC mobility outputs by dataset
# ---------------------------------------------------------------------

for dataset_type in retained_tlc_datasets:

    sample_output_file = sorted(
        TLC_MOBILITY_OUTPUT_DIR.glob(
            f"{dataset_type}_tripdata_*_mobility.parquet"
        )
    )[0]

    print("=" * 80)
    print(dataset_type.upper())
    print("Sample file:", sample_output_file.name)
    print("=" * 80)

    sample_mobility_df = duckdb.sql(f"""
        SELECT *
        FROM read_parquet('{sample_output_file.as_posix()}')
        LIMIT 20
    """).df()

    display(sample_mobility_df)

YELLOW
Sample file: yellow_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,yellow,148,232,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,1.080000,413.000000
1,yellow,164,229,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,6,1.743333,651.666667
2,yellow,246,61,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,9.420000,2719.000000
3,yellow,263,142,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.970000,588.666667
4,yellow,229,137,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,9,1.053333,286.666667
5,yellow,132,225,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,4,10.742500,1871.750000
6,yellow,164,79,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.833333,693.666667
7,yellow,142,151,2022-12-31,2022,12,6,23,weekend_evening,pre_cp,1,2.380000,471.000000
8,yellow,234,144,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,1.886667,666.000000
9,yellow,232,255,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,2.840000,869.000000


GREEN
Sample file: green_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,green,129,226,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,4.020,1650.333333
1,green,134,98,2023-01-01,2023,1,0,1,weekend_overnight,pre_cp,1,5.250,529.000000
2,green,179,7,2023-01-01,2023,1,0,1,weekend_overnight,pre_cp,1,0.620,213.000000
3,green,95,95,2023-01-01,2023,1,0,2,weekend_overnight,pre_cp,1,1.330,390.000000
4,green,260,260,2023-01-01,2023,1,0,2,weekend_overnight,pre_cp,1,0.550,245.000000
5,green,226,196,2023-01-01,2023,1,0,13,weekend_midday,pre_cp,1,3.340,862.000000
6,green,22,227,2023-01-01,2023,1,0,13,weekend_midday,pre_cp,1,0.000,1079.000000
7,green,74,263,2023-01-01,2023,1,0,15,weekend_midday,pre_cp,4,2.010,654.500000
8,green,74,239,2023-01-01,2023,1,0,15,weekend_midday,pre_cp,1,3.230,758.000000
9,green,75,151,2023-01-01,2023,1,0,16,weekend_pm_peak,pre_cp,1,1.030,241.000000


FHVHV
Sample file: fhvhv_tripdata_2023-01_mobility.parquet


,dataset_type,pickup_zone_id,dropoff_zone_id,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,trip_count,avg_trip_distance,avg_trip_duration_seconds
0,fhvhv,24,243,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,5.868667,1071.666667
1,fhvhv,211,74,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,7.808000,1316.000000
2,fhvhv,178,165,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,3,0.893333,328.000000
3,fhvhv,162,179,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,1,4.277000,1402.000000
4,fhvhv,170,246,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,7,1.998286,1058.571429
5,fhvhv,235,69,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,11,1.932273,759.454545
6,fhvhv,101,265,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,6,14.800000,1456.000000
7,fhvhv,256,4,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,10,2.742700,952.200000
8,fhvhv,49,97,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,8,0.947500,358.125000
9,fhvhv,260,173,2023-01-01,2023,1,0,0,weekend_overnight,pre_cp,5,2.188000,835.000000


Findings\. The sample outputs look exactly like what we were hoping to create\. Rather than carrying hundreds of millions of individual trip records through the remainder of the pipeline, we now have compact mobility tables where each row represents an hourly origin\-destination mobility state\. The aggregation appears to be working correctly, with trip counts, average distances, and average trip durations being calculated for each Pickup Taxi Zone × Dropoff Taxi Zone × Date × Hour combination\.

Just as importantly, the process successfully standardized the retained Yellow Taxi, Green Taxi, and FHVHV datasets into a common structure\. Despite originating from different TLC systems, all three datasets now share the same spatial, temporal, and mobility fields, which will make downstream integration and analysis much simpler\. We also intentionally dropped many of the trip\-level operational and economic fields, such as fares, tolls, surcharges, and payment information\. While those fields may be useful for economic analyses, they are not central to the mobility\-focused goals of this project\. The resulting tables retain the information needed for clustering, anomaly detection, mobility regime analysis, and policy evaluation while dramatically reducing the amount of data that must be carried forward\.


Now that we have sample outputs, let’s quantify the practical payoff of aggregating early\. Instead of carrying the full trip\-level parquet files forward, we now have compact mobility\-state tables\. This check compares the raw source file sizes against the post\-aggregation outputs so we can see how much data volume we removed before downstream modeling\.

In [9]:
# ---------------------------------------------------------------------
# Compare raw TLC file sizes to mobility table outputs
# ---------------------------------------------------------------------

size_comparison_records = []

for dataset_type in retained_tlc_datasets:

    source_files = sorted(
        TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
    )

    output_files = sorted(
        TLC_MOBILITY_OUTPUT_DIR.glob(
            f"{dataset_type}_tripdata_*_mobility.parquet"
        )
    )

    source_size_mb = sum(
        path.stat().st_size / (1024 * 1024)
        for path in source_files
    )

    output_size_mb = sum(
        path.stat().st_size / (1024 * 1024)
        for path in output_files
    )

    size_comparison_records.append({
        "dataset_type": dataset_type,
        "source_files": len(source_files),
        "output_files": len(output_files),
        "source_size_mb": source_size_mb,
        "output_size_mb": output_size_mb,
        "size_reduction_mb": source_size_mb - output_size_mb,
        "size_reduction_pct": (
            1 - (output_size_mb / source_size_mb)
            if source_size_mb > 0 else None
        ),
        "compression_ratio": (
            source_size_mb / output_size_mb
            if output_size_mb > 0 else None
        ),
    })

size_comparison_df = pd.DataFrame(size_comparison_records)

size_comparison_df

,dataset_type,source_files,output_files,source_size_mb,output_size_mb,size_reduction_mb,size_reduction_pct,compression_ratio
0,yellow,40,40,2302.932751,488.498317,1814.434434,0.787880,4.714311
1,green,39,39,49.605245,12.808610,36.796635,0.741789,3.872805
2,fhvhv,39,39,18187.889832,3094.771687,15093.118146,0.829844,5.876973


Findings\. The aggregation step produced a substantial reduction in data volume across all retained TLC datasets\. Yellow Taxi shrank from approximately 2\.3 GB to 478 MB, Green Taxi from 50 MB to 12 MB, and FHVHV from 18\.2 GB to 3\.0 GB\. Overall, the resulting mobility tables are roughly 4–6 times smaller than the original trip\-level datasets while preserving the spatial, temporal, and mobility characteristics needed for downstream analysis\. These results validate the decision to aggregate the TLC data during standardization rather than carrying nearly one billion trip records forward into later stages of the pipeline\.

## 1\.2\.3\.2 TLC Temporal Validation

The mobility tables are built around a small set of derived temporal dimensions, including date, year, month, day\_of\_week, hour, temporal\_bucket, and pre\_post\_cp\. These fields drive nearly all of the downstream analysis, so it's worth taking a few minutes to confirm that they were populated correctly during the aggregation process\. Here we'll check for missing values, confirm that the expected temporal categories are present, and review the resulting distributions to make sure the retained TLC datasets provide complete and consistent temporal coverage across the study period\.

## Missingness and count validation of temporal fields

In [10]:
# ---------------------------------------------------------------------
# Validate TLC temporal fields
# ---------------------------------------------------------------------


RUN_TLC_TEMPORAL_FIELD_VALIDATION = False

tlc_mobility_files = sorted(
    TLC_MOBILITY_OUTPUT_DIR.glob("*_mobility.parquet")
)

temporal_fields = [
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "temporal_bucket",
    "pre_post_cp",
]

if (
    RUN_TLC_TEMPORAL_FIELD_VALIDATION
    or not TLC_TEMPORAL_VALIDATION_MANIFEST_PATH.exists()
):

    temporal_validation_records = []

    for field in temporal_fields:

        query = f"""
            SELECT
                '{field}' AS field,
                SUM(CASE WHEN "{field}" IS NULL THEN trip_count ELSE 0 END) AS missing_trip_count,
                SUM(trip_count) AS total_trip_count,
                COUNT(DISTINCT "{field}") AS unique_values
            FROM read_parquet(
                '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
            )
        """

        result = duckdb.sql(query).df().iloc[0].to_dict()

        result["missing_rate"] = (
            result["missing_trip_count"] / result["total_trip_count"]
            if result["total_trip_count"] > 0
            else None
        )

        temporal_validation_records.append(result)

    tlc_temporal_validation_df = pd.DataFrame(
        temporal_validation_records
    )

    tlc_temporal_validation_df.to_csv(
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH,
        index=False
    )

    print(
        "Wrote TLC temporal validation manifest:",
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

else:

    tlc_temporal_validation_df = pd.read_csv(
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

    print(
        "Loaded existing TLC temporal validation manifest:",
        TLC_TEMPORAL_VALIDATION_MANIFEST_PATH
    )

tlc_temporal_validation_df[
    [
        "field",
        "missing_trip_count",
        "missing_rate",
        "unique_values",
    ]
]

Loaded existing TLC temporal validation manifest: pipeline_data/1.2.3.intermediate_tables/tlc_temporal_field_validation_manifest.csv


,field,missing_trip_count,missing_rate,unique_values
0,date,0.0,0.0,1228
1,year,0.0,0.0,12
2,month,0.0,0.0,12
3,day_of_week,0.0,0.0,7
4,hour,0.0,0.0,24
5,temporal_bucket,0.0,0.0,10
6,pre_post_cp,0.0,0.0,2


Findings\. The temporal validation results look clean\. None of the derived temporal fields contain missing values, which confirms that the aggregation process successfully assigned a valid timestamp\-derived representation to every retained TLC trip\. The resulting mobility tables provide complete coverage across all expected months, days of the week, hours, temporal buckets, and congestion\-pricing periods\.

The only result that warrants a closer look is the year field, which reports 12 unique values rather than the expected four years spanning 2023–2026\. This suggests that one or more files may contain unexpected year values or that a validation issue exists in the aggregation process\. We'll investigate that anomaly before closing out the TLC temporal QA\. Aside from that item, the temporal fields appear complete and ready for downstream mobility\-state analysis\.

In [11]:
# ---------------------------------------------------------------------
# Inspect year values
# ---------------------------------------------------------------------

duckdb.sql(f"""
    SELECT
        year,
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

,year,mobility_states,trips
0,2001,7,7.0
1,2002,19,20.0
2,2003,6,6.0
3,2007,1,1.0
4,2008,39,39.0
5,2009,46,47.0
6,2014,1,1.0
7,2022,29,29.0
8,2023,97404458,271075505.0
9,2024,99943903,280736404.0


Findings\. The temporal validation identified a very small number of records with year values outside the project's 2023–2026 study period\. These records account for fewer than 200 trips out of more than 923 million retained trips and are therefore not expected to have any meaningful impact on downstream analysis\. As a result, they were retained and no additional cleaning was performed\.

### Temporal bucket distributions

In [12]:
# ---------------------------------------------------------------------
# Temporal bucket distribution
# ---------------------------------------------------------------------

tlc_temporal_bucket_distribution = duckdb.sql(f"""
    SELECT
        temporal_bucket,
        SUM(trip_count) AS trip_count,
        SUM(trip_count) * 1.0
            / SUM(SUM(trip_count)) OVER () AS trip_share
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_temporal_bucket_distribution

,temporal_bucket,trip_count,trip_share
0,weekday_am_peak,118027769.0,0.128137
1,weekday_evening,134408302.0,0.145920
2,weekday_midday,178104233.0,0.193359
3,weekday_overnight,58066437.0,0.063040
4,weekday_pm_peak,150892768.0,0.163816
5,weekend_am_peak,28879477.0,0.031353
6,weekend_evening,58201930.0,0.063187
7,weekend_midday,78861574.0,0.085616
8,weekend_overnight,54187717.0,0.058829
9,weekend_pm_peak,61478613.0,0.066744


Findings\. The temporal bucket distribution looks reasonable and aligns with what we'd expect from a citywide mobility dataset\. Most trips occur during weekdays, with weekday midday, weekday overnight, and weekday PM peak periods accounting for the largest shares of activity\. Weekend travel patterns are also well represented, although at lower volumes than weekdays\. More importantly, all eight temporal buckets are populated, confirming that the aggregation process successfully captured mobility activity across the full range of expected day and time combinations\. This balanced coverage should provide a solid foundation for downstream clustering, anomaly detection, and mobility regime analysis\.

In [13]:
# ---------------------------------------------------------------------
# Pre/post congestion pricing distribution
# ---------------------------------------------------------------------

tlc_pre_post_distribution = duckdb.sql(f"""
    SELECT
        pre_post_cp,
        SUM(trip_count) AS trip_count,
        SUM(trip_count) * 1.0
            / SUM(SUM(trip_count)) OVER () AS trip_share
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_pre_post_distribution

,pre_post_cp,trip_count,trip_share
0,post_cp,366455445.0,0.397842
1,pre_cp,554653375.0,0.602158


Findings\. Both congestion\-pricing periods are well represented in the resulting mobility tables\. Approximately 60% of trips fall within the pre\-congestion pricing period and roughly 40% fall within the post\-congestion pricing period\. This distribution is consistent with the study timeframe and confirms that the pre\_post\_cp field was assigned correctly during aggregation\. The strong representation of both periods should provide sufficient coverage for future comparative analyses examining how congestion pricing may have influenced mobility patterns across the city\.

## 1\.2\.3\.3 Spatial Validation

The mobility tables are built around pickup and dropoff Taxi Zones, so the next step is to confirm that the aggregation preserved the expected spatial coverage\. Here we'll examine pickup and dropoff zone coverage and verify that the resulting mobility tables continue to represent activity across the full Taxi Zone system\.

### Coverage validation

First, let's check whether  zone fields are complete and broad enough to support citywide analysis\. Here we validate missingness and unique zone coverage for both pickup and dropoff zones, then attach the Taxi Zone reference layer so the results are readable by borough and neighborhood name instead of raw zone IDs\.

In [14]:
# ---------------------------------------------------------------------
# Load Taxi Zone reference layer
# ---------------------------------------------------------------------

TAXI_ZONES_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"

taxi_zones_lookup = (
    pd.read_parquet(TAXI_ZONES_PATH)
    [["location_id", "borough", "zone"]]
    .drop_duplicates()
    .rename(columns={
        "location_id": "taxi_zone_id",
        "borough": "taxi_zone_borough",
        "zone": "taxi_zone_name",
    })
)

taxi_zones_lookup.head()

,taxi_zone_id,taxi_zone_borough,taxi_zone_name
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


In [15]:
# ---------------------------------------------------------------------
# Test 1: Pickup/dropoff Taxi Zone coverage
# ---------------------------------------------------------------------

tlc_zone_coverage = duckdb.sql(f"""
    SELECT
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips,

        SUM(
            CASE
                WHEN pickup_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_pickup_zone_trips,

        SUM(
            CASE
                WHEN dropoff_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_dropoff_zone_trips,

        COUNT(DISTINCT pickup_zone_id) AS unique_pickup_zones,
        COUNT(DISTINCT dropoff_zone_id) AS unique_dropoff_zones
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
""").df()

tlc_zone_coverage["missing_pickup_zone_rate"] = (
    tlc_zone_coverage["missing_pickup_zone_trips"]
    / tlc_zone_coverage["trips"]
)

tlc_zone_coverage["missing_dropoff_zone_rate"] = (
    tlc_zone_coverage["missing_dropoff_zone_trips"]
    / tlc_zone_coverage["trips"]
)

tlc_zone_coverage

,mobility_states,trips,missing_pickup_zone_trips,missing_dropoff_zone_trips,unique_pickup_zones,unique_dropoff_zones,missing_pickup_zone_rate,missing_dropoff_zone_rate
0,327863727,921108820.0,0.0,0.0,263,264,0.0,0.0


Findings\. The mobility tables retain essentially complete Taxi Zone coverage, with no missing pickup or dropoff zones introduced during aggregation\. Coverage remains consistent with the source\-level TLC QA, suggesting that the transition from trip\-level records to mobility states preserved the underlying spatial structure of the data\.

In [16]:
# ---------------------------------------------------------------------
# Test 1b: Pickup/dropoff Taxi Zone coverage by dataset
# ---------------------------------------------------------------------

tlc_zone_coverage_by_dataset = duckdb.sql(f"""
    SELECT
        dataset_type,
        COUNT(*) AS mobility_states,
        SUM(trip_count) AS trips,

        SUM(
            CASE
                WHEN pickup_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_pickup_zone_trips,

        SUM(
            CASE
                WHEN dropoff_zone_id IS NULL
                THEN trip_count ELSE 0
            END
        ) AS missing_dropoff_zone_trips,

        COUNT(DISTINCT pickup_zone_id) AS unique_pickup_zones,
        COUNT(DISTINCT dropoff_zone_id) AS unique_dropoff_zones
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

tlc_zone_coverage_by_dataset["missing_pickup_zone_rate"] = (
    tlc_zone_coverage_by_dataset["missing_pickup_zone_trips"]
    / tlc_zone_coverage_by_dataset["trips"]
)

tlc_zone_coverage_by_dataset["missing_dropoff_zone_rate"] = (
    tlc_zone_coverage_by_dataset["missing_dropoff_zone_trips"]
    / tlc_zone_coverage_by_dataset["trips"]
)

tlc_zone_coverage_by_dataset

,dataset_type,mobility_states,trips,missing_pickup_zone_trips,missing_dropoff_zone_trips,unique_pickup_zones,unique_dropoff_zones,missing_pickup_zone_rate,missing_dropoff_zone_rate
0,fhvhv,272681609,778288736.0,0.0,0.0,263,264,0.0,0.0
1,green,1639425,2091302.0,0.0,0.0,257,260,0.0,0.0
2,yellow,53542693,140728782.0,0.0,0.0,263,262,0.0,0.0


Findings\. Spatial coverage remains strong across all three retained TLC datasets\. Yellow Taxi and FHVHV cover essentially the entire Taxi Zone system, while Green Taxi covers slightly fewer zones, which is expected given its smaller footprint\. One item worth following up on is the additional dropoff zone observed in the aggregated data, which appears to persist from the source datasets and will be investigated further during harmonization QA\.

In [17]:
# ---------------------------------------------------------------------
# Identify pickup zone IDs not found in the Taxi Zone reference layer
# ---------------------------------------------------------------------

official_taxi_zone_ids = set(
    taxi_zones_lookup["taxi_zone_id"]
)

pickup_zone_usage = duckdb.sql(f"""
    SELECT
        dataset_type,
        pickup_zone_id AS zone_id,
        SUM(trip_count) AS trips
    FROM read_parquet(
        '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
    )
    GROUP BY 1, 2
""").df()

unmatched_pickup_zone_usage = (
    pickup_zone_usage[
        ~pickup_zone_usage["zone_id"].isin(official_taxi_zone_ids)
    ]
    .sort_values(["zone_id", "dataset_type"])
)

unmatched_pickup_zone_usage

,dataset_type,zone_id,trips
657,fhvhv,57,177279.0
288,green,57,336.0
181,yellow,57,1646.0
259,fhvhv,105,132.0
596,yellow,105,24.0
477,fhvhv,264,5.0
86,green,264,1333.0
334,yellow,264,552602.0
241,fhvhv,265,33802.0
691,green,265,1165.0


Findings\. Spatial validation identified four pickup zone identifiers that do not currently resolve against the staged Spatial Reference Layer: 57, 105, 264, and 265\. Zone IDs 57, 264, and 265 appear across all three TLC datasets, while Zone 105 appears only in Yellow Taxi and FHVHV\. The fact that these identifiers appear consistently across multiple TLC modalities suggests they are legitimate location codes rather than isolated data\-quality issues introduced during aggregation\.

In [18]:
# ---------------------------------------------------------------------
# Summarize unmatched pickup zone IDs
# ---------------------------------------------------------------------

total_pickup_trips = pickup_zone_usage["trips"].sum()

unmatched_pickup_zone_summary = (
    unmatched_pickup_zone_usage
    .groupby("zone_id", dropna=False)
    .agg(
        trips=("trips", "sum"),
        datasets=("dataset_type", lambda x: ", ".join(sorted(x.unique()))),
    )
    .reset_index()
    .sort_values("zone_id")
)

unmatched_pickup_zone_summary["trip_share"] = (
    unmatched_pickup_zone_summary["trips"]
    / total_pickup_trips
)

unmatched_pickup_zone_summary

,zone_id,trips,datasets,trip_share
0,57,179261.0,"fhvhv, green, yellow",1.946144e-04
1,105,156.0,"fhvhv, yellow",1.693611e-07
2,264,553940.0,"fhvhv, green, yellow",6.013839e-04
3,265,64871.0,"fhvhv, green, yellow",7.042708e-05


Findings\. Although four pickup zone identifiers did not resolve against the current Spatial Reference Layer, their overall impact is minimal\. Together, these zones account for approximately 880 thousand trips, or less than 0\.1% of all retained TLC trips\. Zone 264 contributes the largest share of unmatched activity, followed by Zones 57 and 265, while Zone 105 appears only rarely\. These results indicate that spatial coverage remains effectively complete, with only a very small fraction of TLC activity requiring reconciliation with the Spatial Reference Layer\.

## 1\.2\.3\.4 Metric Validation

Before treating the TLC mobility tables as analysis\-ready, we need to validate the derived mobility metrics themselves\. Temporal fields can be present and zone IDs can map correctly, but impossible values can still slip through from the raw trip records and distort downstream averages, speeds, and temporal features\. This section checks for invalid duration and distance values after aggregation so we can catch upstream data\-quality issues before they flow into harmonization, merging, or feature engineering\.

In [19]:
# ---------------------------------------------------------------------
# Validate TLC mobility metric quality
# ---------------------------------------------------------------------

TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_mobility_metric_quality_manifest.csv"
)

if (
    RUN_TLC_MOBILITY_METRIC_QUALITY_VALIDATION
    or not TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH.exists()
):

    query = f"""
        SELECT
            dataset_type,

            COUNT(*) AS row_count,

            SUM(CASE WHEN avg_trip_duration_seconds < 0 THEN 1 ELSE 0 END)
                AS negative_duration_rows,

            SUM(CASE WHEN avg_trip_duration_seconds = 0 THEN 1 ELSE 0 END)
                AS zero_duration_rows,

            SUM(CASE WHEN avg_trip_duration_seconds > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
                AS excessive_duration_rows,

            SUM(CASE WHEN avg_trip_distance < 0 THEN 1 ELSE 0 END)
                AS negative_distance_rows,

            SUM(CASE WHEN avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
                AS excessive_distance_rows,

            SUM(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                     AND (avg_trip_distance / (avg_trip_duration_seconds / 3600.0)) > {MAX_VALID_TLC_TRIP_SPEED_MPH}
                    THEN 1 ELSE 0
                END
            ) AS excessive_speed_rows,

            MIN(avg_trip_duration_seconds) AS min_avg_trip_duration_seconds,
            MAX(avg_trip_duration_seconds) AS max_avg_trip_duration_seconds,

            MIN(avg_trip_distance) AS min_avg_trip_distance,
            MAX(avg_trip_distance) AS max_avg_trip_distance,

            MIN(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                    THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                    ELSE NULL
                END
            ) AS min_implied_avg_trip_speed,

            MAX(
                CASE
                    WHEN avg_trip_duration_seconds > 0
                    THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                    ELSE NULL
                END
            ) AS max_implied_avg_trip_speed

        FROM read_parquet(
            '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet'
        )

        GROUP BY dataset_type
        ORDER BY dataset_type
    """

    tlc_mobility_metric_quality_df = duckdb.sql(query).df()

    tlc_mobility_metric_quality_df.to_csv(
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH,
        index=False
    )

    print(
        "Wrote TLC mobility metric quality manifest:",
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

else:

    tlc_mobility_metric_quality_df = pd.read_csv(
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

    print(
        "Loaded existing TLC mobility metric quality manifest:",
        TLC_MOBILITY_METRIC_QUALITY_MANIFEST_PATH
    )

display(tlc_mobility_metric_quality_df)

Loaded existing TLC mobility metric quality manifest: pipeline_data/1.2.3.intermediate_tables/tlc_mobility_metric_quality_manifest.csv


,dataset_type,row_count,negative_duration_rows,zero_duration_rows,excessive_duration_rows,negative_distance_rows,excessive_distance_rows,excessive_speed_rows,min_avg_trip_duration_seconds,max_avg_trip_duration_seconds,min_avg_trip_distance,max_avg_trip_distance,min_implied_avg_trip_speed,max_implied_avg_trip_speed
0,fhvhv,272681609,0.0,0.0,0.0,0.0,0.0,0.0,60.0,72373.0,0.0,100.00,0.0,98.919196
1,green,1639425,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86397.0,0.0,99.28,0.0,99.911392
2,yellow,53542693,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86399.0,0.0,99.91,0.0,100.000000


In [20]:
# ---------------------------------------------------------------------
# Assert TLC mobility metric quality
# ---------------------------------------------------------------------

invalid_metric_rows = (
    tlc_mobility_metric_quality_df[
        [
            "negative_duration_rows",
            "zero_duration_rows",
            "excessive_duration_rows",
            "negative_distance_rows",
            "excessive_distance_rows",
            "excessive_speed_rows",
        ]
    ]
    .sum()
    .sum()
)

assert invalid_metric_rows == 0, (
    "TLC mobility metric quality validation failed. "
    "Review tlc_mobility_metric_quality_df for invalid duration, distance, or speed values."
)

print("TLC mobility metric quality validation passed.")

TLC mobility metric quality validation passed.


Findings\. The duration\-quality validation passed after regenerating the TLC mobility tables with invalid\-trip filtering enabled\. All retained TLC datasets now contain zero negative durations, zero\-duration trips, excessive durations, and negative distances\. The issue was traced to a small number of invalid trip\-level records that were contaminating aggregated mobility metrics and was resolved upstream before harmonization, panel integration, and feature engineering\.

### Summary

The aggregated TLC mobility tables retain essentially complete spatial coverage of the Taxi Zone system\. No missing pickup or dropoff zones were introduced during aggregation, and the resulting mobility tables continue to represent activity across nearly the entire Taxi Zone universe\. A small number of pickup zone identifiers \(57, 105, 264, and 265\) did not resolve against the current Spatial Reference Layer, but together account for less than 0\.1% of retained trips\. Overall, the results indicate that the aggregation process preserved the spatial structure of the TLC data and produced mobility tables suitable for downstream Taxi Zone\-level analysis\.

### 🔍 \*\*Cross\-modal Coverage Insight\*\*

One of the benefits of combining multiple mobility systems is that each dataset observes a different portion of the city\. Roadway traffic sensors cover only 138 Taxi Zones, reflecting the limited footprint of the traffic monitoring network\. Bus activity spans 252 Taxi Zones, providing substantially broader geographic coverage, while TLC activity reaches essentially the entire Taxi Zone system with 263 pickup zones and 264 dropoff zones represented\. Subway activity contributes coverage across 156 Taxi Zones and, while more geographically concentrated than Bus or TLC, provides deep visibility into some of the city's most important transit corridors, employment centers, and commercial districts\. Although no single mobility source provides complete visibility into NYC transportation activity, the combined multimodal dataset offers far more comprehensive geographic coverage than any individual modality alone\. This complementary coverage is one of the primary motivations for constructing an integrated mobility panel and may ultimately improve the robustness of downstream anomaly detection and congestion\-pricing analyses\.

### Aggregation Integrity

The most important check for this notebook is whether aggregation preserved the retained TLC trip volume\. Because invalid\-trip filtering is enabled, the correct comparison is not all raw trips versus aggregated trips\. The correct comparison is valid raw trips after filtering versus summed trip\_count values in the mobility tables\. This confirms that aggregation did not accidentally drop or duplicate trips within the retained valid trip universe\.

In [21]:
# ---------------------------------------------------------------------
# Validate aggregation integrity against retained valid TLC trips
# ---------------------------------------------------------------------

REBUILD_TLC_AGGREGATION_INTEGRITY = False

TLC_AGGREGATION_INTEGRITY_PATH = (
    TLC_INTERMEDIATE_OUTPUT_DIR / "tlc_aggregation_integrity.parquet"
)

if TLC_AGGREGATION_INTEGRITY_PATH.exists() and not REBUILD_TLC_AGGREGATION_INTEGRITY:
    aggregation_integrity_df = pd.read_parquet(TLC_AGGREGATION_INTEGRITY_PATH)
    print(
        "Loaded cached TLC aggregation integrity summary:",
        TLC_AGGREGATION_INTEGRITY_PATH
    )

else:
    aggregation_integrity_records = []

    for dataset_type, config in retained_tlc_datasets.items():

        source_files = sorted(
            TLC_DATA_DIR.glob(f"{dataset_type}_tripdata_*.parquet")
        )

        output_files = sorted(
            TLC_MOBILITY_OUTPUT_DIR.glob(
                f"{dataset_type}_tripdata_*_mobility.parquet"
            )
        )

        distance_col = config["distance_col"]
        duration_seconds_expr = config["duration_seconds_expr"]

        trip_quality_filter = f"""
            {duration_seconds_expr} BETWEEN {MIN_VALID_TLC_TRIP_DURATION_SECONDS}
                AND {MAX_VALID_TLC_TRIP_DURATION_SECONDS}
            AND "{distance_col}" BETWEEN {MIN_VALID_TLC_TRIP_DISTANCE}
                AND {MAX_VALID_TLC_TRIP_DISTANCE}
            AND (
                "{distance_col}" / ({duration_seconds_expr} / 3600.0)
            ) <= {MAX_VALID_TLC_TRIP_SPEED_MPH}
        """

        raw_trip_count = 0
        retained_valid_trip_count = 0

        for source_path in source_files:

            raw_trip_count += duckdb.sql(f"""
                SELECT COUNT(*) AS trips
                FROM read_parquet('{source_path.as_posix()}')
            """).fetchone()[0]

            if FILTER_INVALID_TLC_TRIPS:
                retained_valid_trip_count += duckdb.sql(f"""
                    SELECT COUNT(*) AS trips
                    FROM read_parquet('{source_path.as_posix()}')
                    WHERE {trip_quality_filter}
                """).fetchone()[0]
            else:
                retained_valid_trip_count = raw_trip_count

        mobility_table_trip_count = 0
        mobility_state_count = 0

        for output_path in output_files:
            result = duckdb.sql(f"""
                SELECT
                    COUNT(*) AS mobility_states,
                    SUM(trip_count) AS trips
                FROM read_parquet('{output_path.as_posix()}')
            """).df()

            mobility_state_count += result["mobility_states"].iloc[0]
            mobility_table_trip_count += result["trips"].iloc[0]

        filtered_trip_count = raw_trip_count - retained_valid_trip_count

        aggregation_integrity_records.append({
            "dataset_type": dataset_type,
            "source_files": len(source_files),
            "output_files": len(output_files),
            "raw_trip_count": raw_trip_count,
            "retained_valid_trip_count": retained_valid_trip_count,
            "filtered_trip_count": filtered_trip_count,
            "filtered_trip_pct": (
                filtered_trip_count / raw_trip_count
                if raw_trip_count > 0 else None
            ),
            "mobility_table_trip_count": mobility_table_trip_count,
            "trip_count_difference_vs_retained_valid": (
                mobility_table_trip_count - retained_valid_trip_count
            ),
            "mobility_state_count": mobility_state_count,
            "compression_ratio_rows": (
                retained_valid_trip_count / mobility_state_count
                if mobility_state_count > 0 else None
            ),
        })

    aggregation_integrity_df = pd.DataFrame(aggregation_integrity_records)

    float_cols = aggregation_integrity_df.select_dtypes(include=["float"]).columns
    aggregation_integrity_df[float_cols] = (
        aggregation_integrity_df[float_cols]
        .round(6)
    )

    aggregation_integrity_df.to_parquet(
        TLC_AGGREGATION_INTEGRITY_PATH,
        index=False,
    )

    print(
        "Wrote TLC aggregation integrity summary:",
        TLC_AGGREGATION_INTEGRITY_PATH
    )

assert (
    aggregation_integrity_df["trip_count_difference_vs_retained_valid"]
    .eq(0)
    .all()
), (
    "TLC aggregation integrity failed: mobility table trip counts do not match retained valid raw trip counts."
)

aggregation_integrity_df

,dataset_type,source_files,output_files,raw_trip_count,retained_valid_trip_count,filtered_trip_count,filtered_trip_pct,mobility_table_trip_count,trip_count_difference_vs_retained_valid,mobility_state_count,compression_ratio_rows
0,yellow,40,40,143080418,140728782,2351636,0.016436,140728782.0,0.0,53542693,2.628347
1,green,39,39,2160506,2091302,69204,0.032031,2091302.0,0.0,1639425,1.275631
2,fhvhv,39,39,778424569,778288736,135833,0.000174,778288736.0,0.0,272681609,2.854203


Findings\. The retained\-valid integrity check passed\. The mobility\-table trip totals match the raw TLC trips that survived the duration, distance, and implied\-speed filters, with no remaining aggregation gap against the valid\-trip universe\. This confirms the tables are not losing valid trips during aggregation and are no longer being judged against trips that were intentionally filtered out\.

The results also show how much compression the aggregation creates after filtering\. The raw TLC files contain hundreds of millions of valid trip records, while the mobility tables collapse those trips into a much smaller set of origin\-destination mobility states\. The compression ratio should be interpreted within the retained valid trip universe, not the full unfiltered source universe\.

### Top X Sanity Check

In [22]:

# ---------------------------------------------------------------------
# Top pickup zones by modality
# ---------------------------------------------------------------------

for dataset_type in retained_tlc_datasets:

    print("=" * 80)
    print(dataset_type.upper())
    print("=" * 80)

    top_pickup_zones = duckdb.sql(f"""
        SELECT
            pickup_zone_id,
            SUM(trip_count) AS trips
        FROM read_parquet(
            '{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/{dataset_type}_tripdata_*_mobility.parquet'
        )
        GROUP BY 1
        ORDER BY trips DESC
        LIMIT 20
    """).df()

    top_pickup_zones = top_pickup_zones.merge(
        taxi_zones_lookup,
        left_on="pickup_zone_id",
        right_on="taxi_zone_id",
        how="left"
    )

    display(
        top_pickup_zones[
            [
                "pickup_zone_id",
                "taxi_zone_name",
                "taxi_zone_borough",
                "trips",
            ]
        ]
    )

YELLOW


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,237,Upper East Side South,Manhattan,6407335.0
1,132,JFK Airport,Queens,6396252.0
2,161,Midtown Center,Manhattan,6299762.0
3,236,Upper East Side North,Manhattan,5729571.0
4,162,Midtown East,Manhattan,4665452.0
5,186,Penn Station/Madison Sq West,Manhattan,4599320.0
6,230,Times Sq/Theatre District,Manhattan,4530441.0
7,142,Lincoln Square East,Manhattan,4330830.0
8,138,LaGuardia Airport,Queens,4172372.0
9,170,Murray Hill,Manhattan,3930814.0


GREEN


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,74,East Harlem North,Manhattan,481021.0
1,75,East Harlem South,Manhattan,289030.0
2,166,Morningside Heights,Manhattan,105133.0
3,95,Forest Hills,Queens,103590.0
4,41,Central Harlem,Manhattan,97906.0
5,43,Central Park,Manhattan,97226.0
6,82,Elmhurst,Queens,86981.0
7,97,Fort Greene,Brooklyn,67598.0
8,65,Downtown Brooklyn/MetroTech,Brooklyn,58444.0
9,130,Jamaica,Queens,52835.0


FHVHV


,pickup_zone_id,taxi_zone_name,taxi_zone_borough,trips
0,138,LaGuardia Airport,Queens,15148111.0
1,132,JFK Airport,Queens,13708182.0
2,61,Crown Heights North,Brooklyn,10160318.0
3,79,East Village,Manhattan,10092541.0
4,230,Times Sq/Theatre District,Manhattan,9429966.0
5,161,Midtown Center,Manhattan,9219654.0
6,231,TriBeCa/Civic Center,Manhattan,8775002.0
7,68,East Chelsea,Manhattan,8526951.0
8,76,East New York,Brooklyn,8518692.0
9,37,Bushwick South,Brooklyn,8501063.0


Findings\. The most active pickup zones align closely with the expected operating patterns of each TLC modality, providing an additional sanity check on the aggregation and spatial assignment process\. Yellow Taxi activity is concentrated in major Manhattan business districts, tourism hubs, and airports; Green Taxi activity is concentrated in Upper Manhattan, Queens, and Brooklyn neighborhoods consistent with its service footprint; and FHVHV activity exhibits a broader geographic distribution spanning airports, Manhattan commercial centers, and high\-density residential neighborhoods throughout Brooklyn and Queens\. No unexpected geographic concentrations were observed, providing additional confidence that the mobility tables accurately represent real\-world TLC travel patterns\.

## Close

This notebook converted the retained TLC trip datasets into a standardized set of Taxi Zone\-level mobility tables suitable for multimodal integration\. The aggregation process reduced hundreds of millions of individual trip records into a much smaller collection of mobility states while preserving all underlying trip activity\.

Validation confirmed that the generated mobility tables contain complete temporal fields, maintain expected spatial coverage, preserve trip counts exactly, and produce geographic demand patterns consistent with known NYC transportation behavior\. The resulting outputs provide a compact and analytically useful representation of TLC activity that can be combined with traffic, bus, subway, and other mobility sources in later stages of the pipeline\.

The next phase of the project will focus on harmonization QA and multimodal integration, ensuring that all mobility datasets share a consistent spatial and temporal framework before construction of the final Taxi Zone × Date × Temporal Bucket analytical panel\.


In [23]:
# ---------------------------------------------------------------------
# Final QA: Validate TLC final output metric quality
# ---------------------------------------------------------------------

con = duckdb.connect()
con.execute("PRAGMA threads=1")
con.execute("PRAGMA memory_limit='8GB'")

tlc_final_output_quality_df = con.execute(f"""
    SELECT
        dataset_type,

        COUNT(*) AS row_count,

        SUM(CASE WHEN avg_trip_distance < 0 THEN 1 ELSE 0 END)
            AS negative_distance_rows,

        SUM(CASE WHEN avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
            AS excessive_distance_rows,

        SUM(CASE WHEN avg_trip_duration_seconds < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
            AS short_duration_rows,

        SUM(CASE WHEN avg_trip_duration_seconds > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
            AS excessive_duration_rows,

        SUM(
            CASE
                WHEN avg_trip_duration_seconds > 0
                 AND (
                    avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                 ) > {MAX_VALID_TLC_TRIP_SPEED_MPH}
                THEN 1 ELSE 0
            END
        ) AS excessive_implied_speed_rows,

        MIN(avg_trip_distance) AS min_avg_trip_distance,
        MAX(avg_trip_distance) AS max_avg_trip_distance,

        MIN(avg_trip_duration_seconds) AS min_avg_trip_duration_seconds,
        MAX(avg_trip_duration_seconds) AS max_avg_trip_duration_seconds,

        MIN(
            CASE
                WHEN avg_trip_duration_seconds > 0
                THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                ELSE NULL
            END
        ) AS min_implied_avg_trip_speed,

        MAX(
            CASE
                WHEN avg_trip_duration_seconds > 0
                THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                ELSE NULL
            END
        ) AS max_implied_avg_trip_speed

    FROM read_parquet('{TLC_MOBILITY_OUTPUT_DIR.as_posix()}/*_mobility.parquet')
    GROUP BY dataset_type
    ORDER BY dataset_type
""").df()

con.close()

float_cols = tlc_final_output_quality_df.select_dtypes(include=["float"]).columns
tlc_final_output_quality_df[float_cols] = (
    tlc_final_output_quality_df[float_cols].round(3)
)

display(tlc_final_output_quality_df)

,dataset_type,row_count,negative_distance_rows,excessive_distance_rows,short_duration_rows,excessive_duration_rows,excessive_implied_speed_rows,min_avg_trip_distance,max_avg_trip_distance,min_avg_trip_duration_seconds,max_avg_trip_duration_seconds,min_implied_avg_trip_speed,max_implied_avg_trip_speed
0,fhvhv,272681609,0.0,0.0,0.0,0.0,0.0,0.0,100.00,60.0,72373.0,0.0,98.919
1,green,1639425,0.0,0.0,0.0,0.0,0.0,0.0,99.28,60.0,86397.0,0.0,99.911
2,yellow,53542693,0.0,0.0,0.0,0.0,0.0,0.0,99.91,60.0,86399.0,0.0,100.000


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>